In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import *
from pyspark.sql.types import *
import uuid

run_id = str(uuid.uuid4())

silver_df = spark.table("silver_supply_chain")

print("Rows:", silver_df.count())

StatementMeta(, ad906153-f4b6-4c88-9447-f00c237d433d, 3, Finished, Available, Finished, False)

Rows: 180519


In [2]:
dq_results = []

StatementMeta(, ad906153-f4b6-4c88-9447-f00c237d433d, 4, Finished, Available, Finished, False)

In [3]:
failed = silver_df.filter(col("order_id").isNull())

failed_count = failed.count()
total_count = silver_df.count()

dq_results.append((
    run_id,
    "Null Order ID",
    "silver_supply_chain",
    total_count,
    failed_count
))

print("Failed:", failed_count)

StatementMeta(, ad906153-f4b6-4c88-9447-f00c237d433d, 5, Finished, Available, Finished, False)

Failed: 0


In [4]:
failed = silver_df.filter(
    col("product_card_id").isNull()
)

failed_count = failed.count()

dq_results.append((
    run_id,
    "Null Product ID",
    "silver_supply_chain",
    total_count,
    failed_count
))

StatementMeta(, ad906153-f4b6-4c88-9447-f00c237d433d, 6, Finished, Available, Finished, False)

In [5]:
failed = silver_df.filter(
    ~col("late_delivery_risk").isin(0,1)
)

failed_count = failed.count()

dq_results.append((
    run_id,
    "Invalid Late Delivery Risk",
    "silver_supply_chain",
    total_count,
    failed_count
))

StatementMeta(, ad906153-f4b6-4c88-9447-f00c237d433d, 7, Finished, Available, Finished, False)

In [6]:
failed = silver_df.filter(
    col("sales") < 0
)

failed_count = failed.count()

dq_results.append((
    run_id,
    "Negative Sales",
    "silver_supply_chain",
    total_count,
    failed_count
))

StatementMeta(, ad906153-f4b6-4c88-9447-f00c237d433d, 8, Finished, Available, Finished, False)

In [7]:
failed = silver_df.filter(
    col("benefit_per_order") < 0
)

failed_count = failed.count()

dq_results.append((
    run_id,
    "Negative Benefit",
    "silver_supply_chain",
    total_count,
    failed_count
))

StatementMeta(, ad906153-f4b6-4c88-9447-f00c237d433d, 9, Finished, Available, Finished, False)

In [8]:
failed = silver_df.filter(
    col("order_item_quantity") <= 0
)

failed_count = failed.count()

dq_results.append((
    run_id,
    "Invalid Quantity",
    "silver_supply_chain",
    total_count,
    failed_count
))

StatementMeta(, ad906153-f4b6-4c88-9447-f00c237d433d, 10, Finished, Available, Finished, False)

In [9]:
failed = silver_df.filter(
    col("shipping_date") < col("order_date")
)

failed_count = failed.count()

dq_results.append((
    run_id,
    "Shipping Before Order",
    "silver_supply_chain",
    total_count,
    failed_count
))

StatementMeta(, ad906153-f4b6-4c88-9447-f00c237d433d, 11, Finished, Available, Finished, False)

In [10]:
dup = (
    silver_df
    .groupBy(
        "order_id",
        "order_item_id"
    )
    .count()
    .filter(col("count") > 1)
)

failed_count = dup.count()

dq_results.append((
    run_id,
    "Duplicate Order Item",
    "silver_supply_chain",
    total_count,
    failed_count
))

StatementMeta(, ad906153-f4b6-4c88-9447-f00c237d433d, 12, Finished, Available, Finished, False)

In [13]:
print(len(dq_results))

for row in dq_results:
    print(len(row), row)

StatementMeta(, ad906153-f4b6-4c88-9447-f00c237d433d, 15, Finished, Available, Finished, False)

8
5 ('9ecce5ff-6b51-4a4b-9b47-5b16463c7717', 'Null Order ID', 'silver_supply_chain', 180519, 0)
5 ('9ecce5ff-6b51-4a4b-9b47-5b16463c7717', 'Null Product ID', 'silver_supply_chain', 180519, 0)
5 ('9ecce5ff-6b51-4a4b-9b47-5b16463c7717', 'Invalid Late Delivery Risk', 'silver_supply_chain', 180519, 0)
5 ('9ecce5ff-6b51-4a4b-9b47-5b16463c7717', 'Negative Sales', 'silver_supply_chain', 180519, 0)
5 ('9ecce5ff-6b51-4a4b-9b47-5b16463c7717', 'Negative Benefit', 'silver_supply_chain', 180519, 33784)
5 ('9ecce5ff-6b51-4a4b-9b47-5b16463c7717', 'Invalid Quantity', 'silver_supply_chain', 180519, 0)
5 ('9ecce5ff-6b51-4a4b-9b47-5b16463c7717', 'Shipping Before Order', 'silver_supply_chain', 180519, 0)
5 ('9ecce5ff-6b51-4a4b-9b47-5b16463c7717', 'Duplicate Order Item', 'silver_supply_chain', 180519, 0)


In [14]:
dq_schema = [
    "run_id",
    "check_name",
    "table_name",
    "total_rows",
    "failed_rows"
]

dq_df = spark.createDataFrame(
    dq_results,
    dq_schema
)

display(dq_df)

StatementMeta(, ad906153-f4b6-4c88-9447-f00c237d433d, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0b9961e9-7437-4a53-a35a-1dd6234cdb88)

In [15]:
(
    dq_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_data_quality_results")
)

StatementMeta(, ad906153-f4b6-4c88-9447-f00c237d433d, 17, Finished, Available, Finished, False)

In [16]:
display(
    spark.table("silver_data_quality_results")
)

StatementMeta(, ad906153-f4b6-4c88-9447-f00c237d433d, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 47a4f136-4aa3-4f07-907c-ce1015f01e4b)